# Week 3 · Task 1 — Cleaning, targets and CV setup (step by step)

This notebook shows, cell by cell, the cleaning and preparation that `splitting.py` performs, with the checks that justified each decision. Every step is written out explicitly and then asserted to match the module's output, so the notebook and `splitting.py` cannot drift apart.

**Needs:** `pandas`, `numpy`, `pyarrow` (for `read_parquet`), and `splitting.py` in the same folder.

In [16]:
import numpy as np, pandas as pd
import splitting as sp

PATH = "daily_features_v2.parquet"      # <- change if the file lives elsewhere
pd.set_option("display.width", 200, "display.max_columns", 40)

## 1. Load and sanity-check the raw panel

In [17]:
raw = pd.read_parquet(PATH)
raw["day"] = sp._to_datetime(raw["day"])             # handles datetime or epoch-integer storage
raw = raw.sort_values(["station_id", "day"]).reset_index(drop=True)

print(raw.shape)
print(raw.dtypes.value_counts())
print("string columns:", raw.select_dtypes(exclude="number").columns.tolist())
print(raw.groupby("station_id")["day"].agg(["min", "max", "count"]))
print("duplicate (district, day) rows:", raw.duplicated(["station_id", "day"]).sum())
print("calendar gaps:", (raw.groupby("station_id")["day"].diff().dropna().dt.days != 1).sum())

(5480, 105)
float64           89
int64             11
str                4
datetime64[us]     1
Name: count, dtype: int64
string columns: ['station_id', 'day', 'day_of_week', 'season', 'aqi_category']
                  min        max  count
station_id                             
downtown   2023-09-01 2026-08-31   1096
helwan     2023-09-01 2026-08-31   1096
new_cairo  2023-09-01 2026-08-31   1096
october    2023-09-01 2026-08-31   1096
shobra     2023-09-01 2026-08-31   1096
duplicate (district, day) rows: 0
calendar gaps: 0


Expected: 5480 rows × 105 columns, 5 districts × 1096 consecutive days (2023-09-01 → 2026-08-31), no duplicates, no gaps.

## 2. Where are the NaNs?

In [18]:
nan_counts = raw.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
print(nan_counts.to_string())

# per district, and which dates are affected
rows_with_nan = raw[raw.isna().any(axis=1)]
print(rows_with_nan.groupby("station_id")["day"].agg(["min", "max", "count"]))

pm10_lag_1d                      5
pm10_lag_2d                     10
pm10_lag_3d                     15
pm10_lag_7d                     35
pm2_5_corrected_lag_1d           5
pm2_5_corrected_lag_2d          10
pm2_5_corrected_lag_3d          15
pm2_5_corrected_lag_7d          35
carbon_monoxide_lag_1d           5
carbon_monoxide_lag_2d          10
carbon_monoxide_lag_3d          15
carbon_monoxide_lag_7d          35
nitrogen_dioxide_lag_1d          5
nitrogen_dioxide_lag_2d         10
nitrogen_dioxide_lag_3d         15
nitrogen_dioxide_lag_7d         35
sulphur_dioxide_lag_1d           5
sulphur_dioxide_lag_2d          10
sulphur_dioxide_lag_3d          15
sulphur_dioxide_lag_7d          35
ozone_lag_1d                     5
ozone_lag_2d                    10
ozone_lag_3d                    15
ozone_lag_7d                    35
pm10_roll_avg_3d                 5
pm10_roll_avg_7d                 5
pm2_5_corrected_roll_avg_3d      5
pm2_5_corrected_roll_avg_7d      5
carbon_monoxide_roll

**Finding:** the "5 / 10 / 15 / 35" in the brief are totals over 5 districts, i.e. 1 / 2 / 3 / 7 NaNs per district. All of them sit in the first 7 days (2023-09-01 → 07) of each district: warm-up values of the lag, rolling-mean and `dust_transport_proxy` columns. Dropping those 7 days per district removes every NaN with no imputation.

## 3. Check that lag / rolling features only use the past
This decides whether the features are valid for t+1, t+2 and t+3 targets.

In [19]:
g = raw.groupby("station_id")["pm2_5_corrected"]
for L in (1, 2, 3, 7):
    exp = g.shift(L)
    ok = np.allclose(raw[f"pm2_5_corrected_lag_{L}d"], exp, equal_nan=True)
    print(f"lag_{L}d equals PM2.5 at t-{L}:", ok)
for w in (3, 7):
    past_only = g.transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    incl_today = g.transform(lambda s: s.rolling(w, min_periods=1).mean())
    print(f"roll_avg_{w}d uses days <= t-1:", np.allclose(raw[f"pm2_5_corrected_roll_avg_{w}d"], past_only, equal_nan=True),
          "| includes today:", np.allclose(raw[f"pm2_5_corrected_roll_avg_{w}d"], incl_today, equal_nan=True))

lag_1d equals PM2.5 at t-1: True
lag_2d equals PM2.5 at t-2: True
lag_3d equals PM2.5 at t-3: True
lag_7d equals PM2.5 at t-7: True
roll_avg_3d uses days <= t-1: True | includes today: False
roll_avg_7d uses days <= t-1: True | includes today: False


All lags and rolling means are strictly past-only, so a row at day *t* contains nothing from day *t+1* or later. That makes the features legal inputs for any of the three horizons.

## 4. What is `is_health_risk`?

In [20]:
print(raw.groupby("aqi_category")["pm2_5_corrected"].agg(["min", "max", "count"]))
print(pd.crosstab(raw["aqi_category"], raw["is_health_risk"]))
print("\npositive rate overall:", round(raw["is_health_risk"].mean(), 4))
print(raw.groupby("station_id")["is_health_risk"].agg(n_pos="sum", rate="mean").round(3))

                                       min         max  count
aqi_category                                                 
Moderate                         13.316167   35.397750   1684
Unhealthy                        55.417417  148.439833   1210
Unhealthy for Sensitive Groups   35.406833   55.372000   2581
Very Unhealthy                  151.782500  168.514000      5
is_health_risk                     0     1
aqi_category                              
Moderate                        1684     0
Unhealthy                          0  1210
Unhealthy for Sensitive Groups  2581     0
Very Unhealthy                     0     5

positive rate overall: 0.2217
            n_pos   rate
station_id              
downtown      341  0.311
helwan        291  0.266
new_cairo     145  0.132
october       136  0.124
shobra        302  0.276


`is_health_risk` = 1 exactly when same-day PM2.5 > 55.4 µg/m³ (Unhealthy or worse). It is a deterministic function of **today's** PM2.5, so using it unshifted would be nowcasting. The forward-looking classification target is therefore `is_health_risk` shifted back by *h* days (`target_risk_{h}d`). Only 4 AQI categories exist (no "Good", no "Hazardous"; Very Unhealthy = 5 days).

## 5. Drop the first 7 days per district

In [21]:
N_DROP = 7
rank = raw.groupby("station_id").cumcount()
clean = raw[rank >= N_DROP].reset_index(drop=True)

assert not clean.isna().any().any(), "NaNs remain"
print("rows dropped:", len(raw) - len(clean))
print(clean.groupby("station_id").agg(rows=("day", "size"), first_day=("day", "min"), last_day=("day", "max")))

# same result as the module
mod_clean, report = sp.drop_nan_head(raw)
assert mod_clean.equals(clean)
report

rows dropped: 35
            rows  first_day   last_day
station_id                            
downtown    1089 2023-09-08 2026-08-31
helwan      1089 2023-09-08 2026-08-31
new_cairo   1089 2023-09-08 2026-08-31
october     1089 2023-09-08 2026-08-31
shobra      1089 2023-09-08 2026-08-31


,rows_before,rows_with_nan_before,rows_after,rows_dropped,first_day,last_day
station_id,,,,,,
downtown,1096,7,1089,7,2023-09-08,2026-08-31
helwan,1096,7,1089,7,2023-09-08,2026-08-31
new_cairo,1096,7,1089,7,2023-09-08,2026-08-31
october,1096,7,1089,7,2023-09-08,2026-08-31
shobra,1096,7,1089,7,2023-09-08,2026-08-31


## 6. Encodings

In [22]:
enc = clean.copy()
enc["station_id_encoded"] = enc["station_id"].map(sp.STATION_MAP)           # 0-4, trees
for name in sp.STATION_MAP:                                                   # one-hot, LSTM/GRU
    enc[f"station_{name}"] = (enc["station_id"] == name).astype(int)
enc["day_of_week_enc"] = enc["day_of_week"].map(sp.DOW_MAP)                 # Mon=0 .. Sun=6
enc["season_enc"] = enc["season"].map(sp.SEASON_MAP)                        # winter 0 .. autumn 3
enc["aqi_class"] = enc["aqi_category"].map(sp.AQI_MAP)                      # 0..3

print(sp.STATION_MAP)
assert enc.drop(columns=[]).equals(sp.encode_columns(clean))
enc[["station_id", "station_id_encoded", "day_of_week", "day_of_week_enc", "season", "season_enc"]].drop_duplicates().head(10)

{'downtown': 0, 'helwan': 1, 'new_cairo': 2, 'october': 3, 'shobra': 4}


,station_id,station_id_encoded,day_of_week,day_of_week_enc,season,season_enc
0,downtown,0,Friday,4,autumn,3
1,downtown,0,Saturday,5,autumn,3
2,downtown,0,Sunday,6,autumn,3
3,downtown,0,Monday,0,autumn,3
4,downtown,0,Tuesday,1,autumn,3
5,downtown,0,Wednesday,2,autumn,3
6,downtown,0,Thursday,3,autumn,3
84,downtown,0,Friday,4,winter,0
85,downtown,0,Saturday,5,winter,0
86,downtown,0,Sunday,6,winter,0


`day_of_week` and `season` are strings in the parquet, so they need an integer version for the models (the original strings are kept for plotting).

## 7. Forward targets (per district) and tail trim

In [23]:
tg = enc.copy()
g = tg.groupby("station_id")
for h in (1, 2, 3):
    tg[f"target_{h}d"] = g["pm2_5_corrected"].shift(-h)          # PM2.5 at t+h
    tg[f"target_risk_{h}d"] = g["is_health_risk"].shift(-h)      # health-risk flag at t+h  (primary classification)
    tg[f"target_aqi_{h}d"] = g["aqi_class"].shift(-h)            # ordinal AQI at t+h        (secondary)

# the last 3 days of each district have no t+3 label. Trim them so t+1/t+2/t+3 use identical rows.
target_cols = [c for c in tg.columns if c.startswith("target_")]
print("rows with a NaN target before trim:", tg[target_cols].isna().any(axis=1).sum(), "(= 3 per district)")
final = tg.dropna(subset=target_cols).reset_index(drop=True)
for h in (1, 2, 3):
    final[f"target_risk_{h}d"] = final[f"target_risk_{h}d"].astype("int64")
    final[f"target_aqi_{h}d"] = final[f"target_aqi_{h}d"].astype("int64")

print(final.shape)
print(final.groupby("station_id").agg(rows=("day", "size"), first_day=("day", "min"), last_day=("day", "max")))

# same result as the module
module_final = sp.prepare_dataset(raw)
pd.testing.assert_frame_equal(final, module_final)
print("notebook output == sp.prepare_dataset(): OK")

rows with a NaN target before trim: 15 (= 3 per district)
(5430, 123)
            rows  first_day   last_day
station_id                            
downtown    1086 2023-09-08 2026-08-28
helwan      1086 2023-09-08 2026-08-28
new_cairo   1086 2023-09-08 2026-08-28
october     1086 2023-09-08 2026-08-28
shobra      1086 2023-09-08 2026-08-28
notebook output == sp.prepare_dataset(): OK


Result: 5 districts × 1086 days = **5430 rows × 123 columns**, no NaNs.

## 8. Leakage / integrity checks

In [24]:
sp.validate(final)

OK  no NaN in features or targets
OK  target_{1,2,3}d == PM2.5 at t+h within each district
OK  PM2.5 lags/rolling means use days <= t-1 only (no look-ahead; valid for t+1..t+3)
OK  default feature set: 101 columns, current-day outcome and all targets excluded
OK  CV: all 5 districts in every train & test fold, expanding train, contiguous test blocks,
    purge gap = horizon (no label overlap), identical test rows for t+1/t+2/t+3


True

## 9. Features and cross-validation folds

In [25]:
feats = sp.feature_columns(final)                                     # default: as specified in the brief
feats_with_today = sp.feature_columns(final, include_current_day_outcome=True)
print(len(feats), "default features;", len(feats_with_today), "with today's PM2.5 / risk / AQI class")
print("excluded from default:", sorted(set(feats_with_today) - set(feats)))

splitter = sp.PanelTimeSeriesSplit(n_splits=5, test_days=135, min_train_days=400, horizon=1)
folds = splitter.describe(final)
folds

101 default features; 104 with today's PM2.5 / risk / AQI class
excluded from default: ['aqi_class', 'is_health_risk', 'pm2_5_corrected']


,fold,train_start,train_end,train_days,gap_days,test_start,test_end,test_days
0,1,2023-09-08,2024-10-21,410,1,2024-10-23,2025-03-06,135
1,2,2023-09-08,2025-03-05,545,1,2025-03-07,2025-07-19,135
2,3,2023-09-08,2025-07-18,680,1,2025-07-20,2025-12-01,135
3,4,2023-09-08,2025-11-30,815,1,2025-12-02,2026-04-15,135
4,5,2023-09-08,2026-04-14,950,1,2026-04-16,2026-08-28,135


The brief's 5 × 180-day test windows with ≥ 400 training days would need 1300 days; only 1086 exist, so the test window is 135 days (5 × 135 + 411 initial train days = 1086). A purge gap equal to the horizon sits between train and test so no training label falls in the test period.

In [26]:
balance = pd.concat([sp.class_balance(final, h).assign(horizon=h) for h in (1, 2, 3)], ignore_index=True)
print("per-fold balance, horizon 1")
print(balance[balance.horizon == 1].round(3).to_string(index=False))
print("\nper-district, whole dataset")
print(sp.class_balance_by_district(final, 1).round(3))

per-fold balance, horizon 1
 fold  n_train  n_test  pos_rate_train  pos_rate_test  n_pos_train  n_pos_test  scale_pos_weight_train  test_pos_downtown  test_pos_helwan  test_pos_new_cairo  test_pos_october  test_pos_shobra  horizon
    1     2050     675           0.213          0.330          437         223                   3.691              0.519            0.400               0.178             0.059            0.496        1
    2     2725     675           0.241          0.178          656         120                   3.154              0.215            0.244               0.096             0.163            0.170        1
    3     3400     675           0.229          0.102          780          69                   3.359              0.170            0.170               0.059             0.044            0.067        1
    4     4075     675           0.208          0.441          849         298                   3.800              0.526            0.593               0.296  

In [27]:
# Example of what Tasks 2-6 do: one fold, classification
f = sp.get_fold(final, fold=0, horizon=1, task="clf")
print(f["X_train"].shape, f["X_test"].shape, "scale_pos_weight =", round(f["scale_pos_weight"], 3))

(2050, 101) (675, 101) scale_pos_weight = 3.691


## 10. Save outputs

In [28]:
final.to_csv("prepared_dataset.csv", index=False)
try:
    final.to_parquet("prepared_dataset.parquet", index=False)
except Exception as e:
    print("parquet not written:", e)
folds.to_csv("cv_folds.csv", index=False)
balance.to_csv("class_balance_per_fold.csv", index=False)